# Retail Intelligence Platform — Business Case

## The Problem

We are a **multi-city retail chain** selling across Electronics, Fashion, Home & Kitchen, Beauty and Sports.

Our data lives in two worlds that never talk to each other:

| World | Examples | Pain today |
| --- | --- | --- |
| **Structured data** | 1,000 sales transactions, 200 customer profiles, 96 inventory snapshots | Churn is noticed only after a customer is already gone |
| **Unstructured data** | 8 vendor invoices in 5 languages, return policy, warranty guide, vendor agreement | Every document is read and keyed in by hand |

### Business questions we cannot answer today

1. **Who is about to stop buying from us?** → we need churn prediction on customer features
2. **What do our vendor documents say?** → we need AI document understanding
3. **How do we ship AI to production — governed, versioned, monitored?** → we need a platform, not a pile of scripts

### Step 1 — Prove it: everything already lives in one catalog

The next cell counts what we have on the platform today — the gold sales tables, the ML feature tables, and the unstructured PDF volume — all as **governed Unity Catalog objects, side by side**. One platform for structured data AND documents.

In [0]:
# One platform, one catalog: everything lives in Unity Catalog (retail_ai_prod)

print("STRUCTURED — gold_data schema")
for table in ["sales_transactions", "inventory_levels", "customer_profiles"]:
    count = spark.table(f"retail_ai_prod.gold_data.{table}").count()
    print(f"  {table:22s} {count:>6,} rows")

print("\nUNSTRUCTURED — retail_documents volume")
import os
for folder in ["invoices", "policies"]:
    files = os.listdir(
        f"/Volumes/retail_ai_prod/unstructured_docs/retail_documents/{folder}"
    )
    print(f"  {folder:22s} {len(files)} PDF documents")

print("\nML FEATURES — features schema")
for table in ["customer_rfm_features", "product_inventory_features"]:
    count = spark.table(f"retail_ai_prod.features.{table}").count()
    print(f"  {table:26s} {count:>6,} rows")

## The Solution — one platform for data + AI

Every problem above is solved by a capability of the same platform:

| # | Notebook | Databricks capability | Business answer |
| --- | --- | --- | --- |
| 1 | `00_Business_Case` (this one) | — | The problem, in one page |
| 2 | `setup_01` + `gold_ingestion` | Unity Catalog, Volumes | Governed landing zone for data AND documents |
| 3 | `03_Feature_Engineering_RFM` | Feature Engineering, MLflow | Churn prediction from RFM features |
| 4 | `04_AI_Functions_Agents` | `ai_extract`, `ai_classify` | Invoices → structured data, policies → routed automatically |
| 5 | `05_Supervisor_Agent` | Supervisor agent pattern | One entry point that routes any request to the right specialist |
| 6 | `06_Unity_AI_Gateway` | Unity Gateway | Governed, metered access to LLMs |
| 7 | `07_Serving_Endpoints` | Model Serving | The churn model as a REST API |
| 8 | `08_MLflow_Registry_Versioning` | MLflow + UC Model Registry | Versioned models with Champion / Challenger aliases |
| 9 | `09_App_with_Endpoint` | Databricks Apps | A web app any store manager can use |

## Expected outcomes

* **Retention:** churn flagged while the customer can still be saved
* **Operations:** vendor invoices keyed in automatically — any language — instead of by hand
* **Governance:** every AI asset (features, models, LLM calls, apps) is a Unity Catalog object with permissions, lineage and audit
* **Speed to production:** model → registry → REST endpoint → app, in minutes not months